# Complex Networks: Section VI Spectral MFPT vs m0

This notebook plots $\langle T(m_0) \rangle$ for the complex-network Section VI spectral implementation (`section_vi_mfpt_vs_m0`) at fixed network mean degree and multiple reset probabilities $r$.

In [ ]:
using Printf
using Plots
using Graphs
using Random

candidates = [abspath(pwd()), abspath(joinpath(pwd(), "..")), abspath(joinpath(pwd(), "..", ".."))]
project_root = nothing
for p in candidates
    if isfile(joinpath(p, "src", "VoterResetting.jl"))
        global project_root = p
        break
    end
end
project_root === nothing && error("Could not locate project root containing src/VoterResetting.jl from pwd=$(pwd())")

include(joinpath(project_root, "src", "VoterResetting.jl"))
const VR = VoterResetting
gr()
println("Loaded VoterResetting from: $(project_root)")

In [ ]:
Random.seed!(1234)

N = 200
mu = 12.0
p_edge = min(1.0, mu / (N - 1))
g = erdos_renyi(N, p_edge)

if !is_connected(g)
    comps = connected_components(g)
    giant = comps[argmax(length.(comps))]
    g = induced_subgraph(g, giant)[1]
    println("Graph not connected: using giant component with N=$(nv(g)).")
end

m0_values = collect(range(0.0, 0.99, length = 120))
r_values = [0.00, 0.01, 0.03, 0.05, 0.10]
max_states = 200_000

println(@sprintf("Graph summary: N=%d, E=%d, mean degree=%.3f", nv(g), ne(g), 2 * ne(g) / nv(g)))

In [ ]:
gap_info = VR.section_vi_spectral_gap(g; max_states = max_states)
println(@sprintf("Section VI spectral gap = %.6e", gap_info.spectral_gap))
println(@sprintf("Asymptotic scale mu2/(N^2*mu1^2) = %.6e", gap_info.asymptotic_gap_scale))
println(@sprintf("Gap / scale = %.6f", gap_info.ratio_gap_to_asymptotic))

In [ ]:
mfpt_by_r = Dict{Float64, Vector{Float64}}()

for r in r_values
    curve = VR.section_vi_mfpt_vs_m0(
        g;
        m0_values = m0_values,
        r = r,
        max_states = max_states,
    )
    mfpt_by_r[r] = curve.mfpt_values
end

println(@sprintf("Computed %d complex spectral MFPT(m0) curves.", length(r_values)))

In [ ]:
p_full = plot(
    xlabel = "m0",
    ylabel = "MFPT(m0)",
    title = "Complex Section VI Spectral MFPT vs m0 (N=$(nv(g)), mean k=$(round(2*ne(g)/nv(g), digits=2)))",
    legend = :topright,
    linewidth = 2.5,
)

for r in r_values
    tt = mfpt_by_r[r]
    mask = isfinite.(tt) .& (tt .> 0.0)
    plot!(p_full, m0_values[mask], tt[mask], label = @sprintf("r=%.3f", r))
end

p_full

In [ ]:
m0_zoom_mask = m0_values .>= 0.5

p_zoom = plot(
    xlabel = "m0",
    ylabel = "MFPT(m0)",
    title = "Complex Section VI Spectral MFPT vs m0 (zoom m0 >= 0.5)",
    legend = :topright,
    linewidth = 2.5,
    yscale = :log10,
)

for r in r_values
    tt = mfpt_by_r[r]
    mask = m0_zoom_mask .& isfinite.(tt) .& (tt .> 0.0)
    plot!(p_zoom, m0_values[mask], tt[mask], label = @sprintf("r=%.3f", r))
end

p_zoom